In [ ]:
import numpy as np
import pandas as pd
from obspy import read

xlsx = "/Users/ramonmargarit/Phd/Projects/Heterogeneities_Mantle/Heterogeneities-project/data/processed/Shallow_processed_RESULTS.xlsx"
sheet = "best_7_bands_fixed_hold0"
MSEED_PATH = "/Users/ramonmargarit/Phd/Projects/Heterogeneities_Mantle/Heterogeneities-project/notebooks/All_Shallow_Moonquakes.mseed"

FC = 5.0
BANDS = np.array([3., 4., 5., 6., 7., 8., 9.])
STARTTIME_TOL_S = 2.0
SCENARIOS = [dict(LOWER_TOL=0.75, MIN_POST=4, K_NEG=0, K_PRE_POS=0)]


def load_excel_long(xlsx, sheet, *, FC, BANDS):
    d = pd.read_excel(xlsx, sheet_name=sheet)
    need = ["starttime", "station", "fc_hz", "t0_dt_mean"]
    missing = [c for c in need if c not in d.columns]
    if missing:
        raise KeyError(f"Missing columns in sheet '{sheet}': {missing}")

    d["station"] = d["station"].astype(str)
    d["fc_hz"] = pd.to_numeric(d["fc_hz"], errors="coerce").astype(float)
    d["starttime_dt"] = pd.to_datetime(d["starttime"], errors="coerce", utc=True)
    d["t0_dt_mean_dt"] = pd.to_datetime(d["t0_dt_mean"], errors="coerce", utc=True)

    if "distance" in d.columns:
        d["distance_deg"] = pd.to_numeric(d["distance"], errors="coerce")
    elif "epi_deg" in d.columns:
        d["distance_deg"] = pd.to_numeric(d["epi_deg"], errors="coerce")
    else:
        d["distance_deg"] = np.nan

    d = d[d["fc_hz"].isin(BANDS)].copy()
    d["event"] = d["starttime_dt"].astype(str) + "__" + d["station"]
    ref = (
        d[d["fc_hz"].eq(FC)][["event", "t0_dt_mean_dt"]]
        .rename(columns={"t0_dt_mean_dt": "t0_fc_dt"})
        .groupby("event", as_index=False)["t0_fc_dt"]
        .min()
    )
    d = d.merge(ref, on="event", how="left")
    d["dt_rel"] = (d["t0_dt_mean_dt"] - d["t0_fc_dt"]).dt.total_seconds()
    d = d[d["dt_rel"].notna() & d["starttime_dt"].notna() & d["t0_dt_mean_dt"].notna()].copy()
    return d[["event", "station", "starttime_dt", "fc_hz", "dt_rel", "distance_deg", "t0_dt_mean_dt"]].rename(columns={"fc_hz": "band"})


def build_event_band_matrix(df_long, *, BANDS):
    return (
        df_long.pivot_table(index="event", columns="band", values="dt_rel", aggfunc="first")
        .reindex(columns=BANDS)
        .sort_index()
    )


def select_events(*, dt_mat, FC, BANDS, MIN_POST, K_NEG, K_PRE_POS=0, LOWER_TOL=0.0):
    post_bands = [b for b in BANDS if b > FC]
    pre_bands = [b for b in BANDS if b < FC]
    keep = []
    for ev in dt_mat.index:
        dt = dt_mat.loc[ev]
        post_vals = dt[post_bands].dropna()
        if len(post_vals) < MIN_POST:
            keep.append(False)
            continue
        if int((post_vals <= LOWER_TOL).sum()) > K_NEG:
            keep.append(False)
            continue
        pre_vals = dt[pre_bands].dropna()
        if int((pre_vals > LOWER_TOL).sum()) > K_PRE_POS:
            keep.append(False)
            continue
        keep.append(True)
    return pd.Series(keep, index=dt_mat.index, name="keep")


def match_traces_to_excel_events(st, df_long, tol_s):
    by_sta = {sta: g.copy() for sta, g in df_long.groupby("station")}
    event_to_trace = {}
    for tr in st:
        sta = str(getattr(tr.stats, "station", "")).strip()
        if not sta or sta not in by_sta:
            continue
        tr_t0 = pd.Timestamp(tr.stats.starttime.datetime, tz="UTC")
        g = by_sta[sta]
        dt = (g["starttime_dt"] - tr_t0).dt.total_seconds().abs()
        j = dt.idxmin()
        if not np.isfinite(dt.loc[j]):
            continue
        if dt.loc[j] <= tol_s:
            ev = g.loc[j, "event"]
            if ev in event_to_trace:
                prev_tr, prev_diff = event_to_trace[ev]
                if dt.loc[j] < prev_diff:
                    event_to_trace[ev] = (tr, float(dt.loc[j]))
            else:
                event_to_trace[ev] = (tr, float(dt.loc[j]))
    return {ev: tr for ev, (tr, _) in event_to_trace.items()}


df_long = load_excel_long(xlsx, sheet, FC=FC, BANDS=BANDS)
dt_mat = build_event_band_matrix(df_long, BANDS=BANDS)
t0_mat = (
    df_long.pivot_table(index="event", columns="band", values="t0_dt_mean_dt", aggfunc="first")
    .reindex(columns=BANDS)
    .sort_index()
)
st = read(MSEED_PATH)
event_to_trace = match_traces_to_excel_events(st, df_long, tol_s=STARTTIME_TOL_S)

selected_rows = []
for cfg in SCENARIOS:
    keep_mask = select_events(
        dt_mat=dt_mat,
        FC=FC,
        BANDS=BANDS,
        MIN_POST=int(cfg["MIN_POST"]),
        K_NEG=int(cfg["K_NEG"]),
        K_PRE_POS=int(cfg["K_PRE_POS"]),
        LOWER_TOL=float(cfg["LOWER_TOL"]),
    )
    kept_events = [ev for ev in keep_mask.index[keep_mask].tolist() if ev in event_to_trace]
    for ev in kept_events:
        station = ev.split("__", 1)[-1]
        t0_fc = df_long.loc[(df_long["event"] == ev) & (df_long["band"] == FC), "t0_dt_mean_dt"]
        distance_deg = df_long.loc[df_long["event"] == ev, "distance_deg"].dropna()
        selected_rows.append({
            "event": ev,
            "station": station,
            "time_utc": t0_fc.iloc[0] if not t0_fc.empty else pd.NaT,
            "epi_deg": float(distance_deg.iloc[0]) if not distance_deg.empty else np.nan,
        })

selected_events_df = (
    pd.DataFrame(selected_rows)
    .drop_duplicates(subset=["event"])
    .sort_values(["epi_deg", "time_utc"], na_position="last")
    .reset_index(drop=True)
)
print(f"Matched {len(event_to_trace)} Excel events to MiniSEED traces.")
print(selected_events_df.to_string(index=False))

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt, hilbert, get_window

RESULTS_DIR = os.path.join(os.getcwd(), "results", "diffusion_onset")
EVENT_FIG_DIR = os.path.join(RESULTS_DIR, "event_envelopes")
os.makedirs(EVENT_FIG_DIR, exist_ok=True)

HALF_BW = 0.5
WIN_S = 5.0
BP_ORDER = 4
NOISE_START = -150.0
NOISE_END = -30.0
SEARCH_START = -20.0
SEARCH_END = 200.0
THRESHOLD_SIGMA = 3.0
MIN_ABOVE_SEC = 5.0
REF_BAND = float(BANDS.min())
RESULTS_CSV = os.path.join(RESULTS_DIR, "diffusion_onset_results.csv")


def compute_bandpass_limits(center_hz, half_width_hz, sampling_hz):
    low_hz = max(center_hz - half_width_hz, 0.001)
    high_hz = min(center_hz + half_width_hz, 0.99 * sampling_hz / 2)
    return low_hz, high_hz


def build_rms_envelope(signal, sampling_hz, low_hz, high_hz, smooth_window_s, filter_order):
    sos = butter(filter_order, [low_hz / (sampling_hz / 2), high_hz / (sampling_hz / 2)], btype="bandpass", output="sos")
    bandpassed = sosfiltfilt(sos, signal)
    analytic_env = np.abs(hilbert(bandpassed))
    nwin = int(round(smooth_window_s * sampling_hz)) | 1
    window = get_window("hann", nwin)
    window = window / window.sum()
    return np.sqrt(np.convolve(analytic_env ** 2, window, mode="same"))


def estimate_noise_stats(time_s, envelope, noise_start, noise_end, min_noise_n=40):
    mask = np.isfinite(time_s) & np.isfinite(envelope) & (envelope > 0) & (time_s >= noise_start) & (time_s <= noise_end)
    if mask.sum() < min_noise_n:
        raise RuntimeError("Not enough points in pre-event noise window")
    noise_vals = envelope[mask]
    return float(np.median(noise_vals)), float(np.std(noise_vals, ddof=0))


def detect_onset_by_noise(time_s, envelope, *, threshold, search_start, search_end, min_above_sec, sampling_hz):
    mask = np.isfinite(time_s) & np.isfinite(envelope) & (time_s >= search_start) & (time_s <= search_end)
    if not np.any(mask):
        raise RuntimeError("No samples in onset search window")
    t_sel = time_s[mask]
    e_sel = envelope[mask]
    above = e_sel >= threshold
    n_consecutive = max(1, int(round(min_above_sec * sampling_hz)))
    run = 0
    for idx, is_above in enumerate(above):
        run = run + 1 if is_above else 0
        if run >= n_consecutive:
            onset_idx = idx - n_consecutive + 1
            return float(t_sel[onset_idx])
    raise RuntimeError("No onset crossing found")


def plot_event_envelopes(event_id, station, epicentral_deg, band_rows, *, search_start, search_end, output_dir):
    nrows = len(band_rows)
    fig, axes = plt.subplots(nrows, 1, figsize=(11, max(2.5 * nrows, 4.0)), sharex=True)
    if nrows == 1:
        axes = np.array([axes])
    for ax, row in zip(axes, band_rows):
        ax.semilogy(row["time_s"], row["envelope"], color="k", lw=1, label="RMS envelope")
        ax.axhline(row["threshold"], color="tab:red", ls="--", lw=1, label="Noise threshold")
        if np.isfinite(row["onset_time_s"]):
            ax.axvline(row["onset_time_s"], color="tab:green", ls="-", lw=1.5, label="Detected onset")
        ax.axvline(0.0, color="tab:blue", ls=":", lw=1)
        ax.axvspan(search_start, search_end, color="tab:blue", alpha=0.08)
        ax.set_xlim(-100, 250)
        ax.set_ylabel(f"{row['fc_Hz']:.1f} Hz")
        ax.grid(True, which="both", alpha=0.25)
        onset_txt = "nan" if not np.isfinite(row["onset_time_s"]) else f"{row['onset_time_s']:.1f} s"
        ax.set_title(f"onset={onset_txt} | thr={row['threshold']:.2e}")
    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        axes[0].legend(loc="upper right")
    axes[-1].set_xlabel("Time since excel_t0 (s)")
    fig.suptitle(f"Diffusion-onset envelopes | {event_id} | station={station} | distance={epicentral_deg:.2f} deg")
    fig.tight_layout()
    safe_event = "".join(ch if ch.isalnum() else "_" for ch in event_id)
    out_path = os.path.join(output_dir, f"{safe_event}_diffusion_onset.png")
    fig.savefig(out_path, dpi=200)
    plt.show()
    return out_path


rows = []
failed_rows = []

for event_idx, row in selected_events_df.iterrows():
    ev = row["event"]
    tr = event_to_trace.get(ev)
    if tr is None:
        failed_rows.append({"event": ev, "fc_Hz": np.nan, "reason": "Missing MiniSEED trace"})
        continue

    tr_start_utc = pd.Timestamp(tr.stats.starttime.datetime, tz="UTC")
    fs = float(tr.stats.sampling_rate)
    signal = tr.data.astype(float)
    t_raw = np.arange(signal.size) / fs
    band_rows = []

    print(f"Processing event {event_idx + 1}/{len(selected_events_df)}: {ev}")

    for fc_band in BANDS:
        try:
            t0_utc = t0_mat.loc[ev, fc_band]
        except KeyError:
            t0_utc = pd.NaT
        if pd.isna(t0_utc):
            failed_rows.append({"event": ev, "fc_Hz": float(fc_band), "reason": "Missing t0_dt_mean"})
            continue

        t0_utc = pd.Timestamp(t0_utc)
        t0_utc = t0_utc.tz_localize("UTC") if t0_utc.tzinfo is None else t0_utc.tz_convert("UTC")
        t0_offset_s = (t0_utc - tr_start_utc).total_seconds()
        time_s = t_raw - t0_offset_s

        fl_hz, fu_hz = compute_bandpass_limits(fc_band, HALF_BW, fs)
        envelope = build_rms_envelope(signal, fs, fl_hz, fu_hz, WIN_S, BP_ORDER)
        if envelope.size != time_s.size:
            i0 = max(0, (time_s.size - envelope.size) // 2)
            time_s = time_s[i0:i0 + envelope.size]

        try:
            noise_median, noise_std = estimate_noise_stats(time_s, envelope, NOISE_START, NOISE_END)
            threshold = noise_median + THRESHOLD_SIGMA * noise_std
            onset_time_s = detect_onset_by_noise(
                time_s,
                envelope,
                threshold=threshold,
                search_start=SEARCH_START,
                search_end=SEARCH_END,
                min_above_sec=MIN_ABOVE_SEC,
                sampling_hz=fs,
            )
        except Exception as exc:
            failed_rows.append({"event": ev, "fc_Hz": float(fc_band), "reason": str(exc)})
            onset_time_s = np.nan
            noise_median = np.nan
            noise_std = np.nan
            threshold = np.nan

        band_rows.append({
            "fc_Hz": float(fc_band),
            "time_s": time_s.copy(),
            "envelope": envelope.copy(),
            "threshold": float(threshold) if np.isfinite(threshold) else np.nan,
            "onset_time_s": float(onset_time_s) if np.isfinite(onset_time_s) else np.nan,
        })
        rows.append({
            "event": ev,
            "station": row["station"],
            "epi_deg": float(row["epi_deg"]),
            "event_time_utc": row["time_utc"],
            "fc_Hz": float(fc_band),
            "noise_median": float(noise_median) if np.isfinite(noise_median) else np.nan,
            "noise_std": float(noise_std) if np.isfinite(noise_std) else np.nan,
            "threshold": float(threshold) if np.isfinite(threshold) else np.nan,
            "onset_time_s": float(onset_time_s) if np.isfinite(onset_time_s) else np.nan,
            "search_start": SEARCH_START,
            "search_end": SEARCH_END,
            "ref_band_hz": REF_BAND,
        })

    if band_rows:
        plot_event_envelopes(ev, row["station"], row["epi_deg"], band_rows, search_start=SEARCH_START, search_end=SEARCH_END, output_dir=EVENT_FIG_DIR)

onset_results_df = pd.DataFrame(rows).sort_values(["event_time_utc", "fc_Hz"]).reset_index(drop=True)
failed_onsets_df = pd.DataFrame(failed_rows)

ref_df = (
    onset_results_df.loc[onset_results_df["fc_Hz"].eq(REF_BAND), ["event", "onset_time_s"]]
    .rename(columns={"onset_time_s": "ref_onset_time_s"})
)
onset_results_df = onset_results_df.merge(ref_df, on="event", how="left")
onset_results_df["delay_s"] = onset_results_df["onset_time_s"] - onset_results_df["ref_onset_time_s"]

onset_results_df.to_csv(RESULTS_CSV, index=False)
summary_cols = ["event", "station", "epi_deg", "fc_Hz", "onset_time_s", "ref_onset_time_s", "delay_s", "threshold"]
print(onset_results_df[summary_cols].to_string(index=False))
if not failed_onsets_df.empty:
    print()
    print("Failed onset picks:")
    print(failed_onsets_df.to_string(index=False))
print()
print(f"Saved onset results to: {RESULTS_CSV}")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

RESULTS_DIR = os.path.join(os.getcwd(), "results", "diffusion_onset")
DIFFUSION_FIT_CSV = os.path.join(RESULTS_DIR, "diffusion_delay_fit.csv")

if onset_results_df.empty:
    raise RuntimeError("Run the onset processing cell before the diffusion-delay fit.")

fit_df = onset_results_df.dropna(subset=["delay_s", "onset_time_s", "ref_onset_time_s"]).copy()
fit_df = fit_df[fit_df["fc_Hz"] > REF_BAND].copy()
if fit_df.empty:
    raise RuntimeError("No valid delay measurements are available above the reference band.")

events = sorted(fit_df["event"].unique())
event_index = {ev: i for i, ev in enumerate(events)}


def delay_model(freq_hz, fractal_D, scale_event):
    return scale_event * (freq_hz ** (fractal_D - 2.0) - REF_BAND ** (fractal_D - 2.0))


def residuals(params, df):
    fractal_D = params[0]
    log_scales = params[1:]
    model = []
    for row in df.itertuples(index=False):
        scale_event = np.exp(log_scales[event_index[row.event]])
        model.append(delay_model(row.fc_Hz, fractal_D, scale_event))
    return np.asarray(model) - df["delay_s"].to_numpy(float)

x0 = np.concatenate(([2.8], np.zeros(len(events))))
lower = np.concatenate(([2.0], np.full(len(events), -20.0)))
upper = np.concatenate(([3.0], np.full(len(events), 20.0)))
result = least_squares(residuals, x0, bounds=(lower, upper), args=(fit_df,), method="trf", max_nfev=4000)
if not result.success:
    raise RuntimeError(result.message)

best_D = float(result.x[0])
log_scales = result.x[1:]
fit_rows = []
for ev in events:
    scale_event = float(np.exp(log_scales[event_index[ev]]))
    ev_df = fit_df.loc[fit_df["event"] == ev].copy()
    ev_df["delay_model_s"] = delay_model(ev_df["fc_Hz"].to_numpy(float), best_D, scale_event)
    ev_df["event_scale"] = scale_event
    fit_rows.append(ev_df)

fit_results_df = pd.concat(fit_rows, ignore_index=True)
fit_results_df.to_csv(DIFFUSION_FIT_CSV, index=False)
print(f"Best-fit fractal dimension D = {best_D:.3f}")
print(fit_results_df[["event", "fc_Hz", "delay_s", "delay_model_s", "event_scale"]].to_string(index=False))

for ev in events:
    ev_df = fit_results_df.loc[fit_results_df["event"] == ev].sort_values("fc_Hz")
    fig, ax = plt.subplots(figsize=(6.5, 4.0))
    ax.plot(ev_df["fc_Hz"], ev_df["delay_s"], "o-", color="k", label="Observed delay")
    ax.plot(ev_df["fc_Hz"], ev_df["delay_model_s"], "o-", color="crimson", label="Diffusion model")
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel(f"Delay relative to {REF_BAND:.1f} Hz (s)")
    ax.set_title(ev)
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()

print()
print(f"Saved diffusion-delay fit results to: {DIFFUSION_FIT_CSV}")